# 5차시 실습 — 데이터 훑어보기와 정리(EDA) (정답)

교육용 가상 데이터(113행 9열)를 결측값 → 중복 → 이상값 순서로 정리하고, 정리 전후를 비교합니다.

## 1단계. 데이터 읽고 숫자 요약 보기

In [1]:
import pandas as pd

df = pd.read_csv("../../data/weekly/week05/week05_dirty_process_data.csv", encoding="utf-8-sig")
print(df.shape)

(113, 9)


In [2]:
print(df.describe())

            온도_섭씨        압력_Pa    가스유량_slm    처리시간_sec        합격여부
count  110.000000   111.000000  110.000000  113.000000  111.000000
mean   304.941298   994.361336   51.350000  119.922124    0.585586
std     33.576979   140.252613   19.933578    5.191128    0.814287
min    291.300000   -42.910520   -5.000000  107.300000   -1.000000
25%    298.100000  1007.850000   48.755000  116.500000    1.000000
50%    300.450000  1013.100000   49.940000  119.800000    1.000000
75%    302.500000  1017.500000   51.377500  123.400000    1.000000
max    557.727366  1033.100000  250.000000  132.300000    1.000000


### 정리 전 값 기억해 두기

In [3]:
before_rows = len(df)
before_temp = df["온도_섭씨"].mean()
before_pressure = df["압력_Pa"].mean()

print(f"정리 전: {before_rows}행, 평균 온도 {before_temp:.1f}, 평균 압력 {before_pressure:.1f}")

정리 전: 113행, 평균 온도 304.9, 평균 압력 994.4


## 2단계. 결측값 세고 처리하기

In [4]:
print(df.isna().sum())

측정시간        0
로트번호        0
설비번호        0
공정명         0
온도_섭씨       3
압력_Pa       2
가스유량_slm    3
처리시간_sec    0
합격여부        2
dtype: int64


### 결과(합격여부)가 빈 행은 지우기

In [5]:
df = df.dropna(subset=["합격여부"])
print(df.shape)

(111, 9)


### 측정값이 빈 칸은 중앙값으로 채우기

In [6]:
for col in ["온도_섭씨", "압력_Pa", "가스유량_slm"]:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

print(df.isna().sum())

측정시간        0
로트번호        0
설비번호        0
공정명         0
온도_섭씨       0
압력_Pa       0
가스유량_slm    0
처리시간_sec    0
합격여부        0
dtype: int64


## 3단계. 중복 행 찾고 지우기

In [7]:
print(df.duplicated().sum())

3


### 중복 행 들여다보기

In [8]:
dup = df[df.duplicated()]
print(dup[["측정시간", "로트번호", "설비번호", "공정명"]])

                 측정시간      로트번호   설비번호 공정명
110  2024-03-30 05:00  LOT-0278  EQ-04  세정
111  2024-04-05 05:00  LOT-0218  EQ-02  포토
112  2024-03-02 04:00  LOT-0132  EQ-02  세정


In [9]:
df = df.drop_duplicates()
print(df.shape)

(108, 9)


## 4단계. 이상값 찾고 들여다보기

In [10]:
hot = df[df["온도_섭씨"] > 400]
minus_pressure = df[df["압력_Pa"] < 0]
minus_gas = df[df["가스유량_slm"] < 0]
big_gas = df[df["가스유량_slm"] >= 200]

print("온도 400 초과:", len(hot), "건")
print("압력 0 미만:", len(minus_pressure), "건")
print("가스유량 0 미만:", len(minus_gas), "건")
print("가스유량 200 이상:", len(big_gas), "건")

온도 400 초과: 2 건
압력 0 미만: 2 건
가스유량 0 미만: 1 건
가스유량 200 이상: 1 건


### 지우기 전에 다른 열과 함께 보기

In [11]:
print(hot[["설비번호", "공정명", "온도_섭씨", "합격여부"]])

     설비번호 공정명       온도_섭씨  합격여부
17  EQ-03  식각  540.315405   1.0
67  EQ-03  증착  557.727366   1.0


### 직접 해보기 — 음수 압력 행 들여다보기

In [12]:
print(minus_pressure[["설비번호", "공정명", "압력_Pa", "합격여부"]])

     설비번호 공정명      압력_Pa  합격여부
37  EQ-04  식각 -42.910520   1.0
79  EQ-02  산화 -26.981234   1.0


## 5단계. 이상값 지우기

In [13]:
df = df[df["온도_섭씨"] <= 400]
df = df[df["압력_Pa"] >= 0]
df = df[df["가스유량_slm"] >= 0]
df = df[df["가스유량_slm"] < 200]
print(df.shape)

(102, 9)


## 6단계. 정리 전후 비교하기

In [14]:
after_temp = df["온도_섭씨"].mean()
after_pressure = df["압력_Pa"].mean()

print(f"행 개수: {before_rows} → {len(df)}")
print(f"평균 온도: {before_temp:.1f} → {after_temp:.1f}")
print(f"평균 압력: {before_pressure:.1f} → {after_pressure:.1f}")

행 개수: 113 → 102
평균 온도: 304.9 → 300.6
평균 압력: 994.4 → 1013.6


### 직접 해보기 — 정리한 뒤 다시 요약해 보기

In [15]:
print(df.describe())

            온도_섭씨        압력_Pa    가스유량_slm    처리시간_sec        합격여부
count  102.000000   102.000000  102.000000  102.000000  102.000000
mean   300.605392  1013.551961   50.017255  119.763725    0.549020
std      3.492870     7.260605    2.125363    5.235688    0.839937
min    291.300000   996.300000   43.620000  107.300000   -1.000000
25%    298.200000  1008.525000   48.755000  116.500000    1.000000
50%    300.550000  1013.300000   49.960000  119.700000    1.000000
75%    302.475000  1017.875000   51.367500  123.350000    1.000000
max    310.900000  1033.100000   54.700000  132.300000    1.000000


### 도전 문제 1 — 공정명 오타 통일 (빠른 학습자용)

In [16]:
print(df["공정명"].value_counts())

공정명
포토      23
세정      22
산화      21
증착      18
식각      15
중착       1
포토공정     1
식각       1
Name: count, dtype: int64


In [17]:
df["공정명"] = df["공정명"].str.strip()

In [18]:
df["공정명"] = df["공정명"].replace({"중착": "증착", "포토공정": "포토"})
print(df["공정명"].value_counts())

공정명
포토    24
세정    22
산화    21
증착    19
식각    16
Name: count, dtype: int64


### 도전 문제 2 — 형식이 다른 측정시간 찾기 (빠른 학습자용)

In [19]:
odd_time = df[df["측정시간"].str.len() != 16]
print(odd_time["측정시간"])

106    2024/03/05
Name: 측정시간, dtype: str


In [20]:
odd_time = df[df["측정시간"].str[4] != "-"]
print(odd_time["측정시간"])

106          2024/03/05
109    03-05-2024 10:00
Name: 측정시간, dtype: str


## 7단계. AI에게 질문하며 더 알아보기
질문 예시는 강의 "AI로 한 걸음 더" 절에 있습니다. 답은 학생마다 다릅니다.

## 8단계. AI에게 코드 생성 요청하고 직접 실행해보기
강의의 "AI가 만들어줄 수 있는 코드 예시"입니다.

In [21]:
import pandas as pd

data = pd.read_csv("../../data/weekly/week05/week05_dirty_process_data.csv", encoding="utf-8-sig")

for col in data.columns:
    count = data[col].isna().sum()
    rate = count / len(data) * 100
    print(f"{col}: 빈칸 {count}개 ({rate:.1f}%)")

측정시간: 빈칸 0개 (0.0%)
로트번호: 빈칸 0개 (0.0%)
설비번호: 빈칸 0개 (0.0%)
공정명: 빈칸 0개 (0.0%)
온도_섭씨: 빈칸 3개 (2.7%)
압력_Pa: 빈칸 2개 (1.8%)
가스유량_slm: 빈칸 3개 (2.7%)
처리시간_sec: 빈칸 0개 (0.0%)
합격여부: 빈칸 2개 (1.8%)


## 9단계. 연습문제 — 나만의 정제 리포트 만들기

In [22]:
raw_df = pd.read_csv("../../data/weekly/week05/week05_dirty_process_data.csv", encoding="utf-8-sig")
temp_missing = raw_df["온도_섭씨"].isna().sum()
dup_count = raw_df.duplicated().sum()

print(f"정제 리포트: 원본 {len(raw_df)}행 → 정리 후 {len(df)}행")
print(f"온도 빈칸 {temp_missing}건, 중복 행 {dup_count}건을 확인했습니다.")

정제 리포트: 원본 113행 → 정리 후 102행
온도 빈칸 3건, 중복 행 3건을 확인했습니다.


## 10단계. AI로 재미있는 미니 프로그램 만들기
강의 예시 코드입니다.

In [23]:
temps = [301.2, 298.5, 310.9, 540.3, 299.9, 291.3, 302.4]

print("이상값 탐정 게임 시작!")
for t in temps:
    if t > 400:
        print(f"{t}도 → 있을 수 없는 값! 기록 오류부터 의심")
    elif t > 305:
        print(f"{t}도 → 용의자! 정상 범위보다 높아 확인 필요")
    elif t < 295:
        print(f"{t}도 → 용의자! 정상 범위보다 낮아 확인 필요")
    else:
        print(f"{t}도 → 정상 범위")

이상값 탐정 게임 시작!
301.2도 → 정상 범위
298.5도 → 정상 범위
310.9도 → 용의자! 정상 범위보다 높아 확인 필요
540.3도 → 있을 수 없는 값! 기록 오류부터 의심
299.9도 → 정상 범위
291.3도 → 용의자! 정상 범위보다 낮아 확인 필요
302.4도 → 정상 범위
